# Phase 6 — Deep 17-step generative masterplanning

Earlier the agent thought **Site → Roads → Buildings → Paths** — buildings first,
circulation patched in after, which produced functionally invalid layouts. This
rebuild gives it a much deeper chain of reasoning:

> **Site → Public Realm Structure → Arrival Hierarchy → Building Hierarchy →
> Frontages → Entrances → Circulation → Fire & Service → Optimization**

run as **17 explicit steps**:

| # | step | # | step |
|---|------|---|------|
| 1 | Read site | 10 | Generate parking |
| 2 | Determine public / private / service **edges** | 11 | Vehicular network |
| 3 | Create **spatial hierarchy** (the armature) | 12 | Pedestrian desire lines |
| 4 | Assign **building importance** | 13 | Generate fire access |
| 5 | Determine building **frontages** | 14 | Run fire constraints |
| 6 | Place buildings | 15 | Evaluate urban-design quality |
| 7 | Generate entrances | 16 | **Optimize** layout |
| 8 | Generate **arrival spaces** | 17 | Output |
| 9 | Generate drop-offs | | |

Buildings now address an abstract **spatial armature** (a primary axis + frontage
lines + tiered open space) laid in the *empty* site; the drivable streets are
engineered onto that same armature afterwards — the urban-design *framework →
blocks → streets* order. The quality gate **penalises dropped program** (placing
fewer buildings can no longer raise the score), and an **optimiser** keeps the
best of several variants. Every step carries a written `reason`.

`generate_masterplan(site_model, program)` runs all 17 steps and returns one
report with every artifact, a per-step `reasoning` log, a structured `steps`
list, and the five-axis-plus-completeness **urban-design score** that accepts or
rejects the layout. Below we run it and reveal the phases one at a time.

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path

workspace_root = Path.cwd().resolve()
candidate_roots = (workspace_root, workspace_root.parent,
                   workspace_root / 'team_04', workspace_root.parent / 'team_04')
TEAM_ROOT = next((p for p in candidate_roots if (p / 'agent').exists()), None)
if TEAM_ROOT is None:
    raise FileNotFoundError('Run from workspace root, team_04/, or team_04/test_notebooks/')
if str(TEAM_ROOT) not in sys.path:
    sys.path.insert(0, str(TEAM_ROOT))
print('TEAM_ROOT:', TEAM_ROOT)

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype+notebook_connected'

from agent.tools.masterplan import generate_masterplan

# ---- colour keys ----------------------------------------------------------
ROLE_COLORS = {'public': '#dc2626', 'service': '#2563eb',
               'residential': '#16a34a', 'courtyard': '#9333ea'}
EDGE_ROLE_COLORS = {'public': '#dc2626', 'service': '#2563eb', 'private': '#16a34a'}
TIER_FILL = {'landmark': 'rgba(124,58,237,0.32)', 'primary': 'rgba(37,99,235,0.28)',
             'secondary': 'rgba(8,145,178,0.25)', 'background': 'rgba(100,116,139,0.22)'}
TIER_LINE = {'landmark': '#7c3aed', 'primary': '#2563eb',
             'secondary': '#0891b2', 'background': '#64748b'}
REALM_FILL = {'forecourt': 'rgba(220,38,38,0.10)', 'central': 'rgba(234,88,12,0.12)',
              'green': 'rgba(22,163,74,0.14)'}

def _xy(pts):
    xs = [p[0] for p in pts] + [pts[0][0]]
    ys = [p[1] for p in pts] + [pts[0][1]]
    return xs, ys

def new_fig(title, site, buildable=None, h=620):
    fig = go.Figure()
    xs, ys = _xy(site)
    fig.add_trace(go.Scatter(x=xs, y=ys, name='Site boundary', mode='lines',
                             line=dict(color='#1d4ed8', width=3), hoverinfo='skip'))
    if buildable:
        bx, by = _xy(buildable)
        fig.add_trace(go.Scatter(x=bx, y=by, name='Buildable envelope', mode='lines',
                                 fill='toself', fillcolor='rgba(34,197,94,0.06)',
                                 line=dict(color='#16a34a', width=1.5, dash='dash'), hoverinfo='skip'))
    fig.update_layout(title=title, height=h,
                      yaxis=dict(scaleanchor='x', scaleratio=1, visible=False),
                      xaxis=dict(visible=False), margin=dict(l=0, r=0, t=44, b=0),
                      plot_bgcolor='#f0f4ff', paper_bgcolor='#f0f4ff',
                      legend=dict(x=1.02, y=1, bgcolor='white', bordercolor='#ccc', borderwidth=1))
    return fig

# ---- public-realm structure (steps 1-3) -----------------------------------
def draw_site_edges(fig, site, edges):
    role_by = {e['side_index']: e['role'] for e in edges['edges']}
    shown = set()
    for s in site['sides']:
        si = s.get('side_index'); role = role_by.get(si, 'private')
        a, b = s['start'], s['end']; c = EDGE_ROLE_COLORS[role]
        fig.add_trace(go.Scatter(x=[a[0], b[0]], y=[a[1], b[1]], mode='lines',
                                 line=dict(color=c, width=7), opacity=0.65,
                                 name=f'{role} edge', legendgroup=role,
                                 showlegend=role not in shown, hoverinfo='skip'))
        shown.add(role)

def draw_realm_tiers(fig, realm):
    for k, poly in (realm.get('open_space') or {}).items():
        if not poly:
            continue
        xs, ys = _xy(poly)
        fig.add_trace(go.Scatter(x=xs, y=ys, mode='lines', fill='toself',
                                 fillcolor=REALM_FILL.get(k, 'rgba(0,0,0,0.05)'),
                                 line=dict(width=0), name=f'{k} realm', hoverinfo='skip'))

def draw_spine(fig, spine):
    s = spine['vehicular_spine']
    fig.add_trace(go.Scatter(x=[p[0] for p in s], y=[p[1] for p in s], name='primary axis',
                             mode='lines', line=dict(color='#ea580c', width=5), hoverinfo='skip'))
    if spine.get('entry_stub'):
        st = spine['entry_stub']
        fig.add_trace(go.Scatter(x=[p[0] for p in st], y=[p[1] for p in st], name='entry stub',
                                 mode='lines', line=dict(color='#ea580c', width=3, dash='dot'), hoverinfo='skip'))
    if spine.get('fire_loop'):
        fx, fy = _xy(spine['fire_loop'])
        fig.add_trace(go.Scatter(x=fx, y=fy, name='fire loop / 2nd frontage', mode='lines',
                                 line=dict(color='#b91c1c', width=2, dash='dash'), hoverinfo='skip'))

# ---- buildings: plain, by typology, by tier, by fire pass -----------------
def draw_buildings(fig, buildings, fire=None):
    fire_by = {b['building_id']: b for b in (fire or {}).get('buildings', [])}
    for b in buildings:
        xs, ys = _xy(b['boundary'])
        passed = fire_by.get(b['building_id'], {}).get('within_reach')
        if passed is True:
            fill, line = 'rgba(34,197,94,0.40)', '#15803d'
        elif passed is False:
            fill, line = 'rgba(239,68,68,0.40)', '#b91c1c'
        else:
            fill, line = 'rgba(71,85,105,0.30)', '#475569'
        fig.add_trace(go.Scatter(x=xs, y=ys, name=b.get('label', b['building_id']),
                                 mode='lines', fill='toself', fillcolor=fill,
                                 line=dict(color=line, width=2), hoverinfo='text',
                                 text=b.get('placement_reason', '')))
        for hole in b.get('holes', []) or []:
            hx, hy = _xy(hole)
            fig.add_trace(go.Scatter(x=hx, y=hy, mode='lines', fill='toself', fillcolor='#f0f4ff',
                                     line=dict(color=line, width=1, dash='dot'), showlegend=False, hoverinfo='skip'))
        cx = sum(p[0] for p in b['boundary']) / len(b['boundary'])
        cy = sum(p[1] for p in b['boundary']) / len(b['boundary'])
        fig.add_annotation(x=cx, y=cy, text=f"{b.get('label', b['building_id'])}", showarrow=False,
                           font=dict(size=10, color='#111'))

def draw_buildings_by_tier(fig, buildings):
    shown = set()
    for b in buildings:
        xs, ys = _xy(b['boundary']); tier = b.get('tier', 'secondary')
        fig.add_trace(go.Scatter(x=xs, y=ys, mode='lines', fill='toself',
                                 fillcolor=TIER_FILL.get(tier, 'rgba(100,116,139,0.22)'),
                                 line=dict(color=TIER_LINE.get(tier, '#64748b'), width=2),
                                 name=f'{tier}', legendgroup=tier, showlegend=tier not in shown,
                                 hoverinfo='text', text=b.get('placement_reason', '')))
        shown.add(tier)
        for hole in b.get('holes', []) or []:
            hx, hy = _xy(hole)
            fig.add_trace(go.Scatter(x=hx, y=hy, mode='lines', fill='toself', fillcolor='#f0f4ff',
                                     line=dict(color=TIER_LINE.get(tier, '#64748b'), width=1, dash='dot'),
                                     showlegend=False, hoverinfo='skip'))
        cx = sum(p[0] for p in b['boundary']) / len(b['boundary'])
        cy = sum(p[1] for p in b['boundary']) / len(b['boundary'])
        fig.add_annotation(x=cx, y=cy, showarrow=False, font=dict(size=9, color='#111'),
                           text=f"{b['building_id']}<br>#{b.get('importance_rank','?')} {b.get('tier','')}")

# ---- arrival hierarchy ----------------------------------------------------
def draw_entries(fig, access):
    seen = set()
    for r in access['roles']:
        color = '#dc2626' if r['role'] == 'main' else '#0891b2'
        fig.add_trace(go.Scatter(x=[r['point'][0]], y=[r['point'][1]], name=f"{r['role']} entry",
                                 mode='markers', marker=dict(size=17, color=color, symbol='triangle-up',
                                 line=dict(color='white', width=1)),
                                 showlegend=r['role'] not in seen, hoverinfo='text', text=r['reason']))
        seen.add(r['role'])

def draw_entrances(fig, orientation):
    shown = set()
    for b in orientation['buildings']:
        for e in b['entrances']:
            ex, ey, _ = e['point']; dx, dy = e['direction']
            c = ROLE_COLORS.get(e['role'], '#111')
            fig.add_annotation(x=ex + dx * 7, y=ey + dy * 7, ax=ex, ay=ey, xref='x', yref='y',
                               axref='x', ayref='y', showarrow=True, arrowhead=2, arrowsize=1.1,
                               arrowwidth=2, arrowcolor=c)
            fig.add_trace(go.Scatter(x=[ex], y=[ey], mode='markers', name=f"{e['role']} entrance",
                                     marker=dict(size=8, color=c, line=dict(color='white', width=1)),
                                     showlegend=e['role'] not in shown, legendgroup=e['role'],
                                     hoverinfo='text', text=e.get('reason', '')))
            shown.add(e['role'])

def draw_arrival(fig, arrival):
    first = True
    for a in arrival.get('arrival_spaces', []):
        xs, ys = _xy(a['boundary'])
        fig.add_trace(go.Scatter(x=xs, y=ys, mode='lines', fill='toself',
                                 fillcolor='rgba(250,204,21,0.32)', line=dict(color='#eab308', width=1.5),
                                 name='arrival forecourt', legendgroup='arr', showlegend=first,
                                 hoverinfo='text', text=a['reason']))
        first = False

def draw_dropoffs(fig, dropoffs):
    first = True
    for d in dropoffs['dropoffs']:
        ep = d['entrance_point']; dp = d['point']
        fig.add_trace(go.Scatter(x=[ep[0], dp[0]], y=[ep[1], dp[1]], mode='lines',
                                 line=dict(color='#0891b2', width=1, dash='dot'), showlegend=False, hoverinfo='skip'))
        fig.add_trace(go.Scatter(x=[dp[0]], y=[dp[1]], mode='markers', name='drop-off',
                                 marker=dict(size=12, color='#0891b2', symbol='diamond'),
                                 showlegend=first, hoverinfo='text', text=d['reason']))
        first = False

# ---- parking, pedestrian, fire --------------------------------------------
def draw_parking(fig, parking):
    first = True
    for z in parking.get('zones', []):
        xs, ys = _xy(z['boundary'])
        fig.add_trace(go.Scatter(x=xs, y=ys, name='parking', mode='lines', fill='toself',
                                 fillcolor='rgba(56,189,248,0.25)', line=dict(color='#0ea5e9', width=1),
                                 legendgroup='park', showlegend=first, hoverinfo='skip'))
        first = False

def draw_pedestrian(fig, ped):
    first = True
    for p in ped.get('paths', []):
        px = [pt[0] for pt in p['polyline']]; py = [pt[1] for pt in p['polyline']]
        fig.add_trace(go.Scatter(x=px, y=py, name='pedestrian route', mode='lines',
                                 line=dict(color='#16a34a', width=2, dash='dash'),
                                 legendgroup='ped', showlegend=first, hoverinfo='skip'))
        first = False

def draw_fire_points(fig, fire_access):
    first = True
    for p in fire_access.get('appliance_points', []):
        if not p.get('point'):
            continue
        pt = p['point']; ok = p.get('within_reach')
        fig.add_trace(go.Scatter(x=[pt[0]], y=[pt[1]], mode='markers',
                                 marker=dict(size=12, color='#b91c1c' if ok else '#000', symbol='x'),
                                 name='appliance standing point', legendgroup='fire', showlegend=first,
                                 hoverinfo='text', text=f"{p['building_id']}: {p.get('distance_m')} m"))
        first = False

def score_bar(score):
    subs = score['sub_scores']
    fig = go.Figure(go.Bar(x=list(subs.values()), y=list(subs.keys()), orientation='h',
                           marker_color=['#16a34a' if v >= 0.6 else '#dc2626' for v in subs.values()],
                           text=[f'{v:.2f}' for v in subs.values()], textposition='auto'))
    verdict = 'ACCEPTED' if score['accepted'] else 'REJECTED'
    fig.update_layout(title=f"Step 15 — urban-design {score['overall']:.2f}/1.0 → {verdict} (threshold {score['threshold']})",
                      height=340, xaxis=dict(range=[0, 1]), margin=dict(l=10, r=10, t=44, b=10),
                      plot_bgcolor='#f8fafc', paper_bgcolor='#f8fafc')
    return fig

print('helpers defined')

# Scene A — irregular site, mixed typologies, step by step

An irregular concave site fronted by **Main Blvd** (south, 24 m) and a **Service
Lane** (west). Program: a U-court, an H-block, an L-wing and an O-court. We call
`generate_masterplan` once, then reveal the layers in pipeline order.

In [ ]:
SITE = {
    'boundary': [[0,0,0],[180,0,0],[180,80,0],[120,80,0],[120,150,0],[0,150,0]],
    'sides': [
        {'side_index':0,'start':[0,0],'end':[180,0],'adjacent_road':{'name':'Main Blvd','hierarchy':'main','width_m':24.0}},
        {'side_index':1,'start':[180,0],'end':[180,80],'adjacent_road':None},
        {'side_index':2,'start':[180,80],'end':[120,80],'adjacent_road':None},
        {'side_index':3,'start':[120,80],'end':[120,150],'adjacent_road':None},
        {'side_index':4,'start':[120,150],'end':[0,150],'adjacent_road':{'name':'Park Walk','hierarchy':'path','width_m':4.0}},
        {'side_index':5,'start':[0,150],'end':[0,0],'adjacent_road':{'name':'Service Ln','hierarchy':'secondary','width_m':6.0}},
    ],
    'roads': {'main_road_side_index': 0},
}
PROGRAM = [
    {'building_id':'B1','label':'U-court','type':'U','area':1300,'storeys':6},
    {'building_id':'B2','label':'H-block','type':'H','area':1400,'storeys':7},
    {'building_id':'B3','label':'L-wing','type':'L','area':1000,'storeys':5},
    {'building_id':'B4','label':'O-court','type':'O','area':1100,'storeys':5},
]
rep = generate_masterplan(SITE, PROGRAM)
SITEB = SITE['boundary']; BLD = rep['margins']['buildable_boundary']
print(rep['summary'])

print("\n--- the agent's 17-step chain of thought ---")
phase = None
for st in rep['steps']:
    if st['phase'] != phase:
        phase = st['phase']; print(f'\n# {phase}')
    print(f"  [{st['n']:>2}] {st['title']:<34} {st['summary']}")

print('\n--- per-element reasoning ---')
for r in rep['reasoning']:
    print(r)

## Steps 1–3 — read site, classify edges, build the public-realm structure

Before any building exists the agent reads the site, **classifies every edge** as
public (red — the address/arrival face), service (blue — loading) or private
(green — the quiet side), then lays the **public-realm structure**: the buildable
envelope, a **primary movement axis** anchored at the public entry, a perimeter
fire loop (the secondary frontage), and the tiered open space (forecourt →
central → green). This armature is what buildings will address.

In [ ]:
fig = new_fig('Steps 1–3 — edges (public/service/private) + public-realm structure (no buildings yet)', SITEB, BLD)
draw_realm_tiers(fig, rep['public_realm'])
draw_site_edges(fig, rep['site'], rep['edges'])
draw_spine(fig, rep['spine'])
draw_entries(fig, rep['access'])
fig.show()
print('site   :', rep['site']['summary'])
print('edges  :', rep['edges']['summary'])
for e in rep['edges']['edges']:
    print(f"   side {e['side_index']}: {e['role']:<8} — {e['reason']}")
print('realm  :', rep['public_realm']['summary'])

## Steps 4–6 — building hierarchy, frontages, then placement

The agent **ranks the program by prominence** (floor area × storeys × a typology
weight) into tiers — landmark / primary / secondary / background — and gives each
a **frontage**: prominent buildings address the primary axis, lesser ones line
the perimeter. Buildings are then placed **most-important-first** into the slot
that addresses its frontage, inside the envelope, clear of the corridor and of
each other. Colour = tier; the label shows each building's importance rank. Hover
for the placement reason.

In [ ]:
fig = new_fig('Steps 4–6 — buildings coloured by importance tier, placed on their frontages', SITEB, BLD)
draw_spine(fig, rep['spine'])
draw_buildings_by_tier(fig, rep['buildings'])
draw_entries(fig, rep['access'])
fig.show()
print('importance:', rep['importance']['summary'])
print('frontages :', rep['frontages']['summary'])
print('placement :', rep['placement']['summary'])
for b in rep['buildings']:
    print(f"  {b['building_id']} ({b['type']}, {b.get('tier')}): {b['placement_reason']}")

## Steps 7–9 — entrances, arrival spaces, drop-offs

Public doors (red) face the arrival armature; service doors (blue) face the quiet
side. The **arrival hierarchy** then carves a **forecourt** (yellow) in front of
each prominent building's public door, sized by tier — a landmark gets a bigger
arrival space than a background block. Every drop-off (cyan diamond) is generated
**from** a public entrance and snapped to the network — never random.

In [ ]:
fig = new_fig('Steps 7–9 — entrances + arrival forecourts + drop-offs', SITEB, BLD)
draw_spine(fig, rep['spine'])
draw_arrival(fig, rep['arrival_spaces'])
draw_buildings(fig, rep['buildings'])
draw_entrances(fig, rep['entrances'])
draw_dropoffs(fig, rep['dropoffs'])
fig.show()
print('arrival  :', rep['arrival_spaces']['summary'])
for a in rep['arrival_spaces']['arrival_spaces']:
    print(f"   {a['building_id']} ({a['tier']}): {a['reason']}")
print('drop-offs:', rep['dropoffs']['summary'])

## Steps 10–12 — parking, then circulation engineered onto the armature

Parking (light blue) serves destinations. Only **now** are the drivable streets
engineered — corridors that connect the entries, drop-offs, buildings and parking,
laid **along the armature** and bending around footprints. Pedestrian desire lines
(dashed green) run entry→door and parking→door while **treating parking as an
obstacle** — street → site → parking → path → entrance.

In [ ]:
fig = new_fig('Steps 10–12 — parking + vehicular network + pedestrian desire lines', SITEB, BLD)
draw_spine(fig, rep['spine'])
draw_parking(fig, rep['parking'])
draw_buildings(fig, rep['buildings'])
draw_pedestrian(fig, rep['pedestrian_circulation'])
draw_entrances(fig, rep['entrances'])
draw_dropoffs(fig, rep['dropoffs'])
fig.show()
print('parking   :', rep['parking']['summary'])
print('vehicular :', rep['vehicular_circulation']['summary'])
print('pedestrian:', rep['pedestrian_circulation']['summary'])
print('parking integration:')
for z in rep['parking_integration']['zones']:
    print(f"  {z['zone_id']}: {z['sequence']}")

## Steps 13–16 — fire access & constraints, quality, optimisation

Fire access is **generated** (appliance standing points, ✕, found on the network
within reach of every building; any building beyond reach yields a fire-lane
*provision*) and then **validated** — buildings shade green when serviceable, plus
a multi-direction egress check. The layout is scored on **seven** axes
(now including `program_completeness` so dropping buildings can't raise the score,
and `public_realm`); below threshold or under 80% placed it is **rejected**.
Finally the optimiser reports the variants it tried and which it kept.

In [ ]:
fire = rep['fire_safety_egress']['fire_access']
fig = new_fig('Steps 13–14 — fire access (green = serviceable) + appliance points', SITEB, BLD)
draw_spine(fig, rep['spine'])
draw_buildings(fig, rep['buildings'], fire=fire)
draw_fire_points(fig, rep['fire_access'])
draw_entries(fig, rep['access'])
fig.show()
print('fire access:', rep['fire_access']['summary'])
if rep['fire_access']['provisions']:
    for p in rep['fire_access']['provisions']:
        print('   provision:', p['reason'])
print('fire reach :', fire['summary'])
print('egress     :', rep['fire_safety_egress']['egress']['summary'])
print('audit      :', rep['audit']['summary'])

score_bar(rep['urban_design']).show()
print(rep['urban_design']['summary'])
print('\noptimiser:', rep['optimization']['summary'])
for t in rep['optimization']['trials']:
    print(f"   variant {t['variant']:<14} sep={t['separation_m']}m  placed={t['placed']}  "
          f"overall={t['overall']}  accepted={t['accepted']}")

# Scene B — constrained urban block: the completeness gate at work

A tight 110 × 90 m block with a deliberately over-stuffed program (six blocks).
The pipeline reserves the structure, ranks the program, and packs what it can —
then **honestly reports what doesn't fit**. Crucially, with the new
`program_completeness` axis and the 80%-placed gate, dropping half the program now
**lowers** the score and **rejects** the layout (previously a half-empty plan
could score *higher* than a full one). This is the behaviour the brief asked for.

In [ ]:
BLOCK = {'boundary': [[0,0,0],[110,0,0],[110,90,0],[0,90,0]],
         'sides': [
            {'side_index':0,'start':[0,0],'end':[110,0],'adjacent_road':{'name':'High St','hierarchy':'main','width_m':18.0}},
            {'side_index':2,'start':[110,90],'end':[0,90],'adjacent_road':{'name':'Back Ln','hierarchy':'secondary','width_m':6.0}}],
         'roads': {'main_road_side_index': 0}}
DENSE = [{'building_id':f'D{i}','label':f'Blk{i}','type':t,'area':a,'storeys':6}
         for i,(t,a) in enumerate([('H',900),('U',900),('X',800),('O',800),('L',700),('T',700)],1)]
rep2 = generate_masterplan(BLOCK, DENSE)
print(rep2['summary'])
print('edges   :', rep2['edges']['summary'])
print('placed  :', [b['building_id'] for b in rep2['buildings']], '| tiers:',
      [(b['building_id'], b.get('tier')) for b in rep2['buildings']])
print('unplaced:', rep2['placement']['unplaced'])
print('completeness:', rep2['urban_design']['program_completeness'],
      '→', 'ACCEPTED' if rep2['urban_design']['accepted'] else 'REJECTED')

fig = new_fig('Scene B — constrained block (tiered buildings + structure)', BLOCK['boundary'],
              rep2['public_realm']['buildable_boundary'], h=520)
draw_realm_tiers(fig, rep2['public_realm'])
draw_site_edges(fig, rep2['site'], rep2['edges'])
draw_spine(fig, rep2['spine'])
draw_parking(fig, rep2['parking'])
draw_buildings_by_tier(fig, rep2['buildings'])
draw_entrances(fig, rep2['entrances'])
draw_dropoffs(fig, rep2['dropoffs'])
fig.show()
score_bar(rep2['urban_design']).show()
print(rep2['urban_design']['summary'])
for c in rep2['audit']['checks']:
    print(f"  {'PASS' if c['pass'] else 'FAIL'}  {c['check']}: {c['detail']}")

# Summary — what the deep 17-step rebuild changes

- **A deeper chain of thought.** *Site → Public Realm Structure → Arrival
  Hierarchy → Building Hierarchy → Frontages → Entrances → Circulation → Fire &
  Service → Optimization*, run as 17 explicit, individually-reasoned steps — not
  the old *Site → Roads → Buildings → Paths*.
- **Edges are classified first.** Every site edge is read as public / service /
  private, so the layout knows its address, its back-of-house and its quiet side
  before anything is placed.
- **A spatial armature, not a single stick.** Step 3 builds the public-realm
  structure (primary axis + perimeter frontage + tiered open space); buildings
  *address* it, and the drivable streets are engineered onto it afterwards —
  framework → blocks → streets.
- **Buildings have a hierarchy.** They are ranked by prominence and given
  frontages; the most important claim the prime, arrival-facing slots, and each
  gets an **arrival forecourt** scaled to its tier.
- **Fire is generated, then validated.** Appliance standing points and fire-lane
  provisions are produced, then reach + multi-direction egress are checked.
- **The quality gate can't be gamed.** Seven axes including
  `program_completeness`; dropping buildings now *lowers* the score, and a layout
  under 80% placed is rejected.
- **It optimises.** The agent composes several variants and keeps the one that
  places the most program at the highest score — a real generate → evaluate →
  keep-best loop, with every decision carrying a written reason.